<a href="https://colab.research.google.com/github/dadelani/NLP_DL_Intro/blob/main/PAIDeF_AI4D_topic_classification_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text classification: MasakhaNEWS [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dadelani/NLP_DL_Intro/blob/main/PAIDeF_AI4D_topic_classification_notebook.ipynb)



## Clone MasakhaNEWS

```
# This is formatted as code
```

 GitHub Repo


In [1]:
!git clone https://github.com/masakhane-io/masakhane-news

Cloning into 'masakhane-news'...
remote: Enumerating objects: 1383, done.
remote: Counting objects: 100% (252/252), done.
remote: Compressing objects: 100% (175/175), done.
remote: Total 1383 (delta 126), reused 164 (delta 76), pack-reused 1131 (from 1)
Receiving objects: 100% (1383/1383), 92.27 MiB | 9.09 MiB/s, done.
Resolving deltas: 100% (504/504), done.
Updating files: 100% (351/351), done.


## Train Naive Bayes Classifier





#### Install packages

In [2]:
import os
import pandas as pd
import tqdm
import numpy as np

from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn import metrics
from sklearn.metrics import accuracy_score,f1_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
import os

##### specific language to train

In [3]:
folder_name = 'masakhane-news/data/'

feature_column = "text" #can also change to headline
label_column = "category"

np.random.seed(42)

languages = ['eng','hau','ibo','pcm','yor']

# select one
language = languages[0]

##### Get the training/test split

In [4]:
print('-------------------------------------------------')
print(f'--------------Working on {language}-----------------')

train_data = pd.read_csv(f'{folder_name}/{language}/train.tsv',sep='\t')
dev_data = pd.read_csv(f'{folder_name}/{language}/dev.tsv',sep='\t')
test_data = pd.read_csv(f'{folder_name}/{language}/dev.tsv',sep='\t')

print(f' Training set size : {train_data.size}   Dev set size: {dev_data.size}')

-------------------------------------------------
--------------Working on eng-----------------
 Training set size : 13236   Dev set size: 1888


###### Get features

In [5]:
all_text_list  = train_data[feature_column].values.tolist()+dev_data[feature_column].values.tolist()

print('[INFO] Sample data \n',all_text_list[:3])

train_text,train_label = train_data[feature_column].values.tolist(),train_data[label_column].values.tolist()
dev_text,dev_label = dev_data[feature_column].values.tolist(),dev_data[label_column].values.tolist()
test_text,test_label = test_data[feature_column].values.tolist(),test_data[label_column].values.tolist()


unique_label = train_data[label_column].unique().tolist()

print('[INFO] Found Labels : ',unique_label)
#

[INFO] Sample data 
 ['Baljit Sethi cannot understand why it is taking so long for the Post Office to pay him compensation.\n"It doesn\'t seem like they\'re interested in doing anything," he says.\nHe is one of thousands of sub-postmasters who lost everything, after the introduction of a faulty IT system made it appear as though money was going missing.\nfficial inquiry into the scandal was looking into the compensation process for a third time on Thursday.\nquiry heard from government, Post Office and legal representatives, after its chairman Sir Wyn Williams said in September he was "disappointed with the apparent lack of substantial progress to date".\nPost Office, which administers the compensation schemes, said it recognised the "importance of ensuring that postmasters receive timely and fair compensation for the failings associated with the Horizon IT system" and pointed to the numbers of people who had already received payments.\ngovernment says "significant progress" has alread

In [6]:
vectorizer = CountVectorizer(analyzer='char_wb',ngram_range=(1, 3))
vectorizer.fit_transform(all_text_list)

X_train = vectorizer.transform(train_text).toarray()
X_dev= vectorizer.transform(dev_text).toarray()
X_test= vectorizer.transform(test_text).toarray()

y_train = []
for i in train_label:
    y_train.append(unique_label.index(i))

y_dev = []
for i in dev_label:
    y_dev.append(unique_label.index(i))

y_test = []
for i in test_label:
    y_test.append(unique_label.index(i))



##### train Multinomial Naive Bayes

In [7]:
print('=======   MultinomialNB   =========')

classifier = MultinomialNB()
classifier.fit(X_train, y_train)

# Predict Class
y_pred = classifier.predict(X_dev)

# Accuracy
accuracy = metrics.accuracy_score(y_dev, y_pred)
f1 = metrics.f1_score(y_dev, y_pred, average='macro')


print(f'acc: {accuracy}     |  f1_score: {f1}')
print(metrics.classification_report(y_dev, y_pred, target_names=unique_label))

if not os.path.exists(f"{language}/MultinomialNB"):
    os.makedirs(f"{language}/MultinomialNB")

acc = metrics.accuracy_score(y_dev, y_pred)
f1 = metrics.f1_score(y_dev, y_pred,average='weighted')
precision = metrics.precision_score(y_dev, y_pred,average='weighted')
recall = metrics.recall_score(y_dev, y_pred,average='weighted')

print(f"f1 = {f1}")
print(f"loss = {None}")
print(f"precision = {precision}")
print(f"recall = {recall}")

=======   MultinomialNB   =========
acc: 0.8411016949152542     |  f1_score: 0.8347739491029028
               precision    recall  f1-score   support

     business       0.86      0.78      0.82        80
entertainment       0.69      0.91      0.79        75
       health       0.90      0.88      0.89        74
   technology       0.71      0.74      0.73        61
       sports       0.99      0.90      0.94       100
     politics       0.88      0.82      0.85        82

     accuracy                           0.84       472
    macro avg       0.84      0.84      0.83       472
 weighted avg       0.85      0.84      0.84       472

f1 = 0.843585681076453
loss = None
precision = 0.8527484362875276
recall = 0.8411016949152542


## Fine-tune a BERT classifier


In [8]:
!huggingface-cli login

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.

    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) y
Token is valid (permission: write).
The token `Davlan` has been saved to /root/.cache/huggingface/stored_tokens
Cannot authenticate through git-credential a

In [9]:
from datasets import load_dataset, get_dataset_config_names


dataset = load_dataset("masakhane/masakhanews", "eng")

print(f"Features: {dataset['train'].column_names}")
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.tsv:   0%|          | 0.00/10.4M [00:00<?, ?B/s]

dev.tsv: 0.00B [00:00, ?B/s]

test.tsv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/3309 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/472 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/948 [00:00<?, ? examples/s]

Features: ['category', 'headline', 'text', 'url']


DatasetDict({
    train: Dataset({
        features: ['category', 'headline', 'text', 'url'],
        num_rows: 3309
    })
    validation: Dataset({
        features: ['category', 'headline', 'text', 'url'],
        num_rows: 472
    })
    test: Dataset({
        features: ['category', 'headline', 'text', 'url'],
        num_rows: 948
    })
})

##### Install the requirements for text-classification from HuggingFace

In [10]:
!pip install -r https://raw.githubusercontent.com/huggingface/transformers/refs/heads/main/examples/pytorch/text-classification/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00


###### download the text classification fine-tuning code

In [11]:
!wget https://raw.githubusercontent.com/huggingface/transformers/refs/heads/main/examples/pytorch/text-classification/run_classification.py

--2025-12-03 12:59:01--  https://raw.githubusercontent.com/huggingface/transformers/refs/heads/main/examples/pytorch/text-classification/run_classification.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 32736 (32K) [text/plain]
Saving to: ‘run_classification.py’

run_classification. 100%[===================>]  31.97K  --.-KB/s    in 0s      

2025-12-03 12:59:01 (153 MB/s) - ‘run_classification.py’ saved [32736/32736]



##### Run code for a specific language

In [12]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [13]:
dataset="masakhane/masakhanews"
subset="eng"

!python run_classification.py \
    --model_name_or_path  google-bert/bert-base-cased \
    --dataset_name "masakhane/masakhanews" \
    --dataset_config_name "eng" \
    --shuffle_train_dataset \
    --metric_name accuracy \
    --text_column_name "headline,text" \
    --text_column_delimiter "\t" \
    --label_column_name category \
    --do_train \
    --do_eval \
    --do_predict \
    --max_seq_length 512 \
    --per_device_train_batch_size 32 \
    --learning_rate 2e-5 \
    --num_train_epochs 3 \
    --output_dir classifier_${subset}

2025-12-03 12:59:25.579508: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764766765.599687     623 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764766765.605685     623 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1764766765.620567     623 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764766765.620594     623 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1764766765.620598     623 computation_placer.cc:177] computation placer alr

## Prompt an LLM e.g. Gemini

In [14]:
import os, asyncio
import glob
import json
import pandas as pd
from pathlib import Path
import ast

#import google.generativeai as genai
from google import genai

from google.genai import types
import glob
import json
from sklearn.metrics import f1_score, accuracy_score
import numpy as np
from collections import Counter
import re

##### Add Gemini Key

In [15]:
GOOGLE_API_KEY = ""

client = genai.Client(api_key=GOOGLE_API_KEY)
model_name = "gemini-2.5-flash"

def chat_completion(message, model_name):
    #response = model.generate_content(message)

    response = client.models.generate_content(
        model=model_name,
        contents=message,
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_budget=128)
            # Turn off thinking:
            # thinking_config=types.ThinkingConfig(thinking_budget=0)
            # Turn on dynamic thinking:
            # thinking_config=types.ThinkingConfig(thinking_budget=-1)
        ),
    )
    return response.text


ValueError: Missing key inputs argument! To use the Google AI API, provide (`api_key`) arguments. To use the Google Cloud API, provide (`vertexai`, `project` & `location`) arguments.

In [ ]:
def getlabel_string(filename):
    with open(filename) as f:
        label_list = f.read().splitlines()
    label_string = label_list[0]
    for i, value in enumerate(label_list[:-2], 1):
        label_string += ', ' + label_list[i]

    label_string += ' or ' + label_list[-1]

    return label_string, label_list

In [ ]:
def news_classification(lang):
    prompt_prefix = 'Is this a piece of news regarding {{"'
    prompt_suffix = '"}}? '
    metric = {}

    df = pd.read_csv("masakhane-news/data/"+lang+'/test.tsv', sep='\t', nrows=20)
    print(df.shape)
    label_string, label_list = getlabel_string("masakhane-news/data/"+lang+"/labels.txt")

    all_input_messages = []
    responses = []
    for index in range(df.shape[0]):  # df.shape[0]
        headline = df['headline'].iloc[index]
        content = df['text'].iloc[index]
        text_string = headline + ' ' + content
        query = ' '.join(text_string.split()[:100])

        message = 'One labels only. ' + prompt_prefix + label_string + prompt_suffix + query

        response = chat_completion(message, model_name)

        responses.append(response)

        if index % 100 == 0:
            print(index, 'processed', response)

    completions = []
    for completion_text in responses:
        completions.append(completion_text)
    df['llm'] = completions

    for label in label_list:
        df['llm'][df['llm'].str.contains(label.lower())] = label

    f1 = f1_score(df['category'], df['llm'], average='weighted')*100
    metric[lang] = f1

    print("Accuracy of "+model_name+" for "+lang, f1)

    return df




In [ ]:
lang = "eng"
df = news_classification(lang)

In [ ]:
df.head(20)

##### Verbalizer to extract results

In [ ]:
random_labels = {'business': 'technology',
                 'entertainment': 'health',
                 'health': 'politics',
                 'politics':'religion',
                 'religion':'sports',
                 'sports': 'technology',
                 'technology': 'business'}

verbalizer = {'business': 'business',
                 'entertainment': 'entertainment',
                 'health': 'health',
                 'politics':'politics',
                 'religion':'religion',
                 'sports': 'sports',
                 'technology': 'technology',
              'finance':'business', 'economy':'business', 'economics':'business', 'political':'politics',
              'music': 'entertainment', 'sport': 'sports'}


def get_predictions(gpt_results, target, categories):

    categories = set(categories) | set(verbalizer.keys())

    predictions = []
    for s, sentence in enumerate(gpt_results):
        try:
            sent_text = re.sub(r'[^a-zA-Z0-9\s]', ' ', sentence.lower())
            #if s>700: print(sent_text)
        except:
            print(s, 'hehe')
            sent_text = 'health'

        set_words = set(sent_text.split())
        inter_words = list(set_words & set(categories))
        label_coming_first = [verbalizer[tok] for tok in sent_text.split() if tok in categories]

        label = 'business'
        if len(inter_words) == 0:
            label = random_labels[target[s]]
        else:
            if len(inter_words) == 1:
                label = verbalizer[inter_words[0]]
            else:
                new_labels = [verbalizer[lab] for lab in inter_words]
                label_count = Counter(new_labels)
                label_n_no = label_count.most_common(1)[0]
                if label_n_no[1] > 1:
                    label = label_n_no[0]
                else:
                    label = label_coming_first[0]
                    #print(new_labels, label, sent_text)
                #label = 'sports'#max(new_labels, key=new_labels.count)

        predictions.append(label)

    return predictions

In [ ]:
target = list(df['category'].values)
llm = list(df['llm'].values)
categories = list(df['category'].unique())

predictions = get_predictions(llm, target, categories)
print(predictions)

res = round(accuracy_score(target, predictions)*100, 1)
print(lang, res)